<center>
  <img src="https://python.langchain.com/assets/images/rag_concepts-4499b260d1053838a3e361fb54f376ec.png"
       width="640" alt="RAG concepts">
  <div><small><a href="https://python.langchain.com/docs/concepts/rag">Source</a></small></div>

# Building a minimal Retrieval-Augmented Generation pipeline

In this tutorial, you will build a simple Retrieval-Augmented Generation pipeline using the [ETH Zurich Degree Programmes PDF](https://ethz.ch/content/dam/ethz/main/education/bachelor/studiengaenge/files/ETH-Zurich-Degree-programmes.pdf) as the corpus. We begin by showing why plain LLM queries on this document don't always work, continuing with setting up the RAG pipeline.

## Preparations
As a first step, we need to install a few libraries:

In [ ]:
%pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.1/948.1 kB 28.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install pypdf


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 70.4 MB/s eta 0:00:00
  Attempting uninstall: zstandard
    Found existing installation: zstandard 0.19.0
    Not uninstalling zstandard at /opt/conda/lib/python3.10/site-packages, outside environment /home/jovyan/torch-venv
    Can't uninstall 'zstandard'. No files were found to uninstall.
  Attempting uninstall: jsonpatch
    Found existing installation: jsonpatch 1.32
    Not uninstalling jsonpatch at /opt/conda/lib/python3.10/site-packages, outside environment /home/jovyan/torch-venv
    Can't uninstall 'jsonpatch'. No files were found to uninstall.

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install sentence_transformers


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install -U ipywidgets  # may need to update ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 52.4 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 4.0.13
    Not uninstalling widgetsnbextension at /opt/conda/lib/python3.10/site-packages, outside environment /home/jovyan/torch-venv
    Can't uninstall 'widgetsnbextension'. No files were found to uninstall.
  Attempting uninstall: jupyterlab_widgets
    Found existing installation: jupyterlab_widgets 3.0.13
    Not uninstalling jupyterlab-widgets at /opt/conda/lib/python3.10/site-packages, outside environment /home/jovyan/torch-venv
    Can't uninstall 'jupyterlab_widgets'. No files were found to uninstall.
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 8.1.5
    Not uninstalling ipywidgets at /opt/conda/lib/python3.10/site-packages, outside environment /home/jovyan/torch-venv
    Can't uninstall 'ipywidgets'. No files were found to uninstall.

[notice] A new release of pip is avai

In [ ]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 33.6 MB/s eta 0:00:0000:0100:01m

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install docling

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 40.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 47.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 912.2/912.2 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 66.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.9 MB/s eta 0:00:00
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=29b180636286726d23ef48400c67327129efb61e40c546f481666a55af3c53

In [ ]:
%pip install accelerate


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


We recommend you restart the kernel so that the newly installed packages will be available.

## Launching models using AzureOpenAI

We will call the LLM via an API (Application Programming Interface) — a defined interface that allows two programs to interact (here, the notebook code and the LLM service). Here, we will use AzureOpenAI.

![](https://i.ibb.co/nsJTYh6j/LLM-azure-text-clean.png)

In [ ]:
import os
from openai import AzureOpenAI

# Technical set-up
azure_key = YOUR_KEY_HERE

endpoint = os.getenv("ENDPOINT_URL", "https://cas-dml-llm.openai.azure.com/")
deployment = os.getenv("DEPLOYMENT_NAME", "gpt-35-turbo")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY", azure_key)

# Initialize AzureOpenAI Service client with key-based authentication
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2024-05-01-preview",
)

We initialized the AzureOpenAI client and now want to send queries to the model. The system message sets the assistant’s behavior (answer in English, be helpful), and the user message contains our actual question.

In [ ]:
# Сore funtion to interact with the llm over the API
def get_ai_response(query):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer questions in English."},  # set the behavior
        {"role": "user", "content": query}  # our question
    ]

    response = client.chat.completions.create(
        model=deployment,  # deployment name in Azure
        messages=messages,
        temperature=0,  # deterministic answer
    )

    return response.choices[0].message.content  # extract AI's reply; choices containts possible model answers

Now, let us check the model:

In [ ]:
# Example query
query = "What is RAG in AI?"
answer = get_ai_response(query)
print(answer)

RAG stands for "Retrieval-Augmented Generation" in the field of artificial intelligence. It is a model that combines retrieval-based and generative approaches to improve the performance of natural language generation tasks. RAG models use a retriever component to search for relevant information from a large corpus of text and then generate responses based on this retrieved information. This approach helps to enhance the quality and relevance of the generated text.


## Why we need RAG?

The test answer seems nice. Now, let's try asking something more local and recent. The question we want to ask is: “How many ETH Zurich spin-off companies were founded in 2024?”.

In [ ]:
query = "How many ETH Zurich spin-off companies were founded in 2024?"
answer = get_ai_response(query)
print(answer)

I'm sorry, but I do not have real-time information on the number of spin-off companies founded in 2024 from ETH Zurich. I recommend checking the official ETH Zurich website or contacting their technology transfer office for the most up-to-date information on spin-off companies.


**Why it happened?** The model doesn't have up-to-date or document-specific knowledge. Remember that the LLM’s knowledge comes from a fixed training snapshot. Therefore, if a fact is niche or very recent, the model likely didn’t see it during the training.

So, let us try to enrich the prompt with relevant information extracted from the file about ETH Zurich so the model can answer based on facts, not guesswork.

## What is RAG (briefly)

* **R**etrieval — fetch relevant chunks from an external corpus (e.g., a PDF).

* **A**ugmented **G**eneration — inject these chunks into the prompt so the LLM relies on facts.

![](https://i.ibb.co/qMghzdwY/image.png)

### Main steps:

1. Extract text from the PDF → split it into chunks.

2. Build vector representations of the chunks using a neural model.

3. Create an index (a vector database).

4. For each question → embed the question → search for the top-k most similar chunks → build a prompt like:

```
Context:
<chunk1>
<chunk2>

Question: <question>

Instructions: answer using only the context; if the answer is not present — say so.
```

5. Send this prompt to the model via AzureOpenAI.


## Prepare the data

### Step 1. Install dependencies and download the PDF

Here, we download the PDF from a URL and then open it with pypdf — a Python library for reading and manipulating PDFs (extract text, merge/split, rotate, etc.).

In [ ]:
# download the document by link

!wget --no-check-certificate \
     --header="User-Agent: Mozilla/5.0 (X11; Linux x86_64)" \
     -O ETH-Zurich-Degree-programmes.pdf \
     "https://ethz.ch/content/dam/ethz/main/education/bachelor/studiengaenge/files/ETH-Zurich-Degree-programmes.pdf"

--2025-09-18 08:33:14--  https://ethz.ch/content/dam/ethz/main/education/bachelor/studiengaenge/files/ETH-Zurich-Degree-programmes.pdf
Resolving ethz.ch (ethz.ch)... 129.132.19.216, 2001:67c:10ec:254::216
Connecting to ethz.ch (ethz.ch)|129.132.19.216|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2329081 (2.2M) [application/pdf]
Saving to: ‘ETH-Zurich-Degree-programmes.pdf’

ETH-Zurich-Degree-p 100%[===================>]   2.22M  --.-KB/s    in 0.06s   

2025-09-18 08:33:14 (40.3 MB/s) - ‘ETH-Zurich-Degree-programmes.pdf’ saved [2329081/2329081]



### Step 2. Read the PDF and extract text

In [ ]:
from pypdf import PdfReader
from pathlib import Path

FILE_PATH = Path("ETH-Zurich-Degree-programmes.pdf")
reader = PdfReader(FILE_PATH)
number_of_pages = len(reader.pages)

entire_text = ""
for page_num in range(number_of_pages):
    page = reader.pages[page_num]
    entire_text += page.extract_text()

# Let us have a look at the text
entire_text[:200]

'Degree programmes\nAn overview of all subject areas2\nHello  \nand welcome  \nto ETH Zurich\n3\nSpitztitel  Lorem Ipsum dolor sit amet\nWe are pleased that you are interested in a study \nprogramme at ETH Zur'

Hidden/template layers often get mixed into the plain text. For instance, "Spitztitel Lorem Ipsum dolor sit amet": German "Spitztitel" = running title; the Lorem Ipsum is template/placeholder text left in a master page layer. Text extractors still see it.

### Step 3. Split text into chunks

Now we’ll split the text into chunks — small, self-contained pieces that the model can search

We’ll use a ```RecursiveCharacterTextSplitter``` splitter, which cuts on natural boundaries (paragraphs → lines → words) and keeps a small overlap to preserve context.

Note: depending on your data, you may wish to pick another splitter: Markdown (```MarkdownHeaderTextSplitter```), token-based (RecursiveTokenTextSplitter) for strict context budgets, or language-aware for code.

In [ ]:
# LangChain is an open-source framework for building LLM apps from modular blocks
# We are going to only take the splitter from it

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

text_chunks = text_splitter.split_text(entire_text)
print(f"Total chunks: {len(text_chunks)}")

Total chunks: 138


```chunk_size``` is the target length of each chunk (in tokens or characters, depending on the splitter), and ```chunk_overlap``` is how much content to repeat between adjacent chunks (e.g., 10–25%) so information that spans a boundary isn’t lost.

**How to choose the chunk size and overlap?** A typical answer should fit in one chunk (sometimes two) without dragging along lots of extra text. Start with defaults, then test 5–10 real queries: if the relevant passage isn’t in the top-k, increase chunk size/overlap; if the context feels bloated, decrease them.

**Larger** ```chunk_size``` generally increases recall — i.e., the chance that relevant content appears in the top-k — because more context stays together; but it also adds noise, slows search, and reduces diversity. **Smaller** ```chunk_size``` improves precision and speed but can split facts across chunks (mitigate with 10–20% overlap or a higher k).”

Use ```chunk_overlap``` to protect **boundary info**: a moderate 10–20% overlap keeps cross-boundary details; higher (20–30%, e.g., for code/step-by-step docs) improves recall but bloats the index and yields near-duplicates; lower (0–10%) is lighter and faster but may miss boundary context.

Let us have a look at the first chunks:

In [ ]:
text_chunks[:2]

['Degree programmes\nAn overview of all subject areas2\nHello  \nand welcome  \nto ETH Zurich\n3\nSpitztitel  Lorem Ipsum dolor sit amet\nWe are pleased that you are interested in a study \nprogramme at ETH Zurich. If you have a flair for \nfigures and a liking for technology, and if you are \ninterested in the natural sciences and the big \nchallenges of the 21st century, then you are heading \nfor the right place. ETH studies are based on well-\nfounded expert knowledge. That, in turn, will provide',
 'you with a basis on which you will be able to stay \nabreast of constant technological and scientific change. \nIn addition, your studies at ETH Zurich will provide \nyou with the skill to reflect on your knowledge in \nvarious contexts. It is not least  because of this that \nour graduates are among the most sought-after \nspecialists and executives in industry and research.\nGünther Dissertori, Rector\n4\nOur degree \nprogrammes\nStudying  \nat ETH Zurich\n Facts and figures\n05 ETH 

## Make vector dataset

## How to compare the strings (what is embeddings)?

Now we have a list of chunks (context passages).

**What we want?** Given a question, we want to automatically retrieve the most relevant chunks and pass them to the model.

**What do we need?** To do that, we need a way to measure the semantic similarity (closeness) between the question and each chunk.

**How to compare the closeness?** Raw strings are hard to compare “by meaning.” So we turn each chunk into a vector of numbers (an **embedding**). Embeddings have a useful property: texts with similar meaning → nearby vectors (high cosine similarity). That lets us search by meaning, not exact words.

**Why we have this property or what is an encoder?** Encoder is a pretrained model that maps text → vector. It’s trained so semantically similar texts land close together in vector space. Important: what counts as “similar” depends on the task, so different tasks may need different encoders.

**How to pick an encoder?** There are [lots of encoders](https://huggingface.co/spaces/mteb/leaderboard)! Choose based on your task and technical needs.

**What we'll do next?** In practice, we convert text to embeddings (vectors) and compare them (e.g., with cosine similarity), then send the top-k chunks to the model.

<center>
  <img src="https://arize.com/wp-content/uploads/2022/06/blog-king-queen-embeddings.jpg"
       width="640" alt="RAG concepts">
  <div><small><a href="https://arize.com/blog-course/embeddings-meaning-examples-and-how-to-compute/">Source</a></small></div>

### Step 1. Chunks to embeddings (vectors)

In [ ]:
%env TOKENIZERS_PARALLELISM=false  # technical

env: TOKENIZERS_PARALLELISM=false  # technical


In [ ]:
from sentence_transformers import SentenceTransformer

# Download a model to create vector representations of text
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Encode text chunks into embeddings (one vector per chunk)
embeddings = model.encode(text_chunks, batch_size=64, show_progress_bar=True)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

### Step 2. Make a vector database

To store embeddings and search for the nearest embeddings fastly, we use [FAISS](https://github.com/facebookresearch/faiss) (Facebook AI Similarity Search). It is a fast library for finding nearest vectors. A **FAISS index** is both a container for your embedding vectors and the search method that makes lookups fast.

Let's make a FAISS index.

In [ ]:
# Create a FAISS index for efficient similarity search

import faiss
embeddings = embeddings.astype("float32")

# Cosine-similarity trick: L2-normalize so inner product ≈ cosine similarity
faiss.normalize_L2(embeddings)

# Build an exact inner-product index
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

## Getting the answer

Now, we will do a final step! For each question → embed the question → search for the top-k most similar chunks → build a prompt like:

```
Context:
<chunk1>
<chunk2>

Question: <question>

Instructions: answer using only the context; if the answer is not present — say so.

### Step 1. Embed the question

In [ ]:
query = "How many ETH Zurich spin-off companies were founded in 2024?"

# reminder: model is an embedding model we initialized before
query_embedding = model.encode([query], normalize_embeddings=True)

### Step 2. Search for the top-k most similar chunks

In [ ]:
# reminder: index is FAISS index (vector database) we have made before
D, I = index.search(query_embedding, k=2)  # search for top-2 closest chunks

retrieved_chunks = [text_chunks[i] for i in I[0]]  # I has (n_queries, k) shape; we have 1 query
retrieved_chunks

['Albert  Einstein and  \nWolfgang Pauli) \nwas the year in which \nthe founders of modern \n Switzerland created this \n place of innovation and \nknowledge\nspin-off companies \nfounded in 2024 \nof which CHF 1.42bn \nbasic funding by the \nConfederation\nETH budget:\n6622\nscientists\nand 3344 technical \nand administrative staff\n7ETH Zentrum\nCampus\n2\nETH Hönggerberg9\nFeatures of \nETH Zurich studies\nWhat is special about studying at ETH in comparison with',
 'enough time for sober consideration.\nStart\nGet ready for your ETH journey. You yourself can contribute quite a \nlot to a successful start of your studies. Our diverse services and \nthe checklist for the start of your studies will help you in this.\nApplying\nYou can register or apply for a study programme at                       \nETH Zurich from 1 December. Please note the relevant  \napplication deadlines.\nethz.ch/application-bachelor\nethz.ch/beginning-your-studies\nethz.ch/which-study-programme']

**The chunk we need**. The nearest chunk was retrieved correctly, but the content is low-quality. We need the following fragment, but it’s corrupted by text extraction:
>Switzerland created this \n place of innovation and \nknowledge\nspin-off companies \nfounded in 2024 \nof which CHF 1.42bn

**Why it happened?** The problem is that while the PDF looks fine to a human, the computer fails to capture the number 43 in the extracted text. Let us have a look at the PDF fragment we need:

<center>
  <img src="https://i.ibb.co/pv9j6FzJ/image.png"
       width="640" alt="RAG concepts">



### Get the answer

Finally, we’ll generate an answer by injecting the retrieved chunks into the model prompt. First, let’s refactor the AI interuction function:

In [ ]:
def get_ai_response_with_context(query, context):
    messages = [
        {
            "role": "system",
            "content": "You are an assistant that answers questions based on the provided information."
            },  # set the behavior

        {
            "role": "user",
            "content": f"Question: {query}\n\nContext:\n{context}"
            }
    ]

    response = client.chat.completions.create(
        model=deployment,  # deployment name in Azure
        messages=messages,
        temperature=0,  # deterministic answer
    )

    return response.choices[0].message.content  # extract AI's reply; choices containts possible model answers

Now, we can get an answer:

In [ ]:
query = "How many ETH Zurich spin-off companies were founded in 2024?"
context = "\n\n".join(retrieved_chunks)

answer = get_ai_response_with_context(query, context)
print(answer)

Based on the provided information, it is not specified how many ETH Zurich spin-off companies were founded in 2024.


As we have seen, the reason is data preparation. Let's fix it.

## Another try: data preparation using Docling

### Step 1. Data preparation using Docling

Our goal is to extract text from PDF correctly. We use the [Docling](https://github.com/docling-project/docling?tab=readme-ov-file) library to convert PDF into structured Markdown. Unlike simple text extraction, Docling combines existing text with OCR (text from scanned pages) and preserves basic structure such as headings, paragraphs, and tables.

This is a pretty compute-intensive task, so it will take some time.

In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert("ETH-Zurich-Degree-programmes.pdf")

# the below code can be used to work with pdf files that are available on the internet:
# source = "https://ethz.ch/content/dam/ethz/main/education/bachelor/studiengaenge/files/ETH-Zurich-Degree-programmes.pdf"  # document per local path or URL
# converter = DocumentConverter()
# result = converter.convert(source)

2025-09-18 11:08:45,741 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-09-18 11:08:45,756 - INFO - Going to convert document batch...
2025-09-18 11:08:45,757 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-09-18 11:08:45,767 - INFO - Loading plugin 'docling_defaults'
2025-09-18 11:08:45,768 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-09-18 11:08:45,780 - INFO - Loading plugin 'docling_defaults'
2025-09-18 11:08:45,782 - INFO - Registered ocr engines: ['easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-09-18 11:08:45,808 - INFO - Accelerator device: 'cuda:0'
2025-09-18 11:08:47,427 - INFO - Accelerator device: 'cuda:0'
2025-09-18 11:08:48,153 - INFO - Accelerator device: 'cuda:0'
2025-09-18 11:08:48,515 - INFO - Processing document ETH-Zurich-Degree-programmes.pdf
2025-09-18 11:09:08,404 - INFO - Finished converting document ETH-Zurich-Degree-programmes.pdf in 22.67 sec.


In [ ]:
md = result.document.export_to_markdown()
print(md[:500])

<!-- image -->

<!-- image -->

2

## Hello and welcome to ETH Zurich

<!-- image -->

3

Spitztitel

Lorem Ipsum dolor sit amet

<!-- image -->

We are pleased that you are interested in a study programme at ETH Zurich. If you have a flair for figures and a liking for technology, and if you are interested in the natural sciences and the big challenges of the 21st century, then you are heading for the right place. ETH studies are based on wellfounded expert knowledge. That, in turn, will provide


As we see, it's much better. Let us clean the text.

## Step 2. Clean the extracted text

The text still contains noise: invisible placeholders, fake headings, and formatting artifacts. This function cleans the text by removing image markers, “lorem ipsum” placeholders, and page artifacts, while also fixing line breaks and hyphenated words.

In [ ]:
import re

def clean_docling_markdown(text: str) -> str:
    # remove comments <!-- image -->
    text = re.sub(r"<!--\s*image\s*-->\s*", "", text, flags=re.I)

    # remove obvious placeholder strings
    placeholders = [
        r"lorem ipsum.*",   # lorem ipsum ...
        r"spitztitel",      # layout headline
        r"dummy",           # the word "dummy"
        r"platzhalter",     # "placeholder" in German
        r"placeholder",     # "placeholder" in English
    ]
    text = re.sub("(?mi)^(" + "|".join(placeholders) + r")\s*$", "", text)

    # normalize line breaks and spaces
    text = text.replace("\r", "")
    text = re.sub(r"[ \t]+\n", "\n", text)              # remove trailing spaces before newline
    text = re.sub(r"\n{3,}", "\n\n", text)              # maximum two consecutive newlines

    # merge word breaks across line breaks: "Spin-\noff" -> "Spinoff"
    text = re.sub(r"(\w)[\-–]\n(\w)", r"\1\2", text)

    # remove very short single-line "placeholder headings"
    text = re.sub(r"(?m)^\s*[A-ZÄÖÜ][A-Za-zÄÖÜäöüß]{1,12}\s*$", "", text)

    return text.strip()

clean_md = clean_docling_markdown(md)
clean_md[:1000]

"2\n\n## Hello and welcome to ETH Zurich\n\n3\n\nWe are pleased that you are interested in a study programme at ETH Zurich. If you have a flair for figures and a liking for technology, and if you are interested in the natural sciences and the big challenges of the 21st century, then you are heading for the right place. ETH studies are based on wellfounded expert knowledge. That, in turn, will provide you with a basis on which you will be able to stay abreast of constant technological and scientific change. In addition, your studies at ETH Zurich will provide you with the skill to reflect on your knowledge in various contexts. It is not least   because of this that our graduates are among the most sought-after specialists and executives in industry and research.\n\nGünther Dissertori, Rector\n\n4\n\n## Contents\n\n## Studying at ETH Zurich\n\n## ETH Zurich - where the future begins\n\nFreedom and personal responsibility, entrepreneurship and cosmopolitanism: Switzerland's values are the

We don’t remove the page numbers here because the PDF’s layout makes it easy to accidentally delete other content. You might normally do this to reduce unnecessary information.

## Step 3. Split the text into chunks

Now, we should just repeat the steps we have done before.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_text(clean_md)

print(f"Total chunks: {len(chunks)}")
print(f"Example chunk: \n {chunks[0]}")

Total chunks: 202
Example chunk: 
 2

## Hello and welcome to ETH Zurich

3


Looks nice, let us repeat the vector dataset preparation.

## Step 4. Encode chunks and create FAISS index

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(chunks, batch_size=64, show_progress_bar=True)

embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

2025-09-18 11:10:11,159 - INFO - Use pytorch device_name: cuda
2025-09-18 11:10:11,160 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

## Step 5. Retrieve top chunks for a query

In [ ]:
query = "How many ETH Zurich spin-off companies were founded in 2024?"
query_embedding = model.encode([query], normalize_embeddings=True)
D, I = index.search(query_embedding, k=2)  # get top-2 closest chunks

retrieved_chunks = [chunks[i] for i in I[0]]
retrieved_chunks

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

['and 3344 technical and administrative staff\n\n37 spin-off companies founded in 2024\n\n22\n\nNobel Laureates (among them Albert   Einstein and Wolfgang Pauli)\n\n2.04bn\n\nof which CHF 1.42bn basic funding by the Confederation\n\nETH budget:\n\nETH Hönggerberg\n\n2\n\nETH Zentrum\n\n## Features of ETH Zurich studies\n\nWhat is special about studying at ETH in comparison with other universities? How much time do you invest in your studies? When do the examinations take place? How is the study programme structured?',
 "## ETH Zurich - where the future begins\n\nFreedom and personal responsibility, entrepreneurship and cosmopolitanism: Switzerland's values are the foundation of ETH Zurich. Here, students find an environment which encourages independent thinking, while researchers find a climate that inspires top achievements. Located in the heart of Europe and connected worldwide, ETH Zurich develops solutions to the global challenges of today and tomorrow.\n\nMore than patent applicat

## Step 6. Ask the LLM with context

In [ ]:
query = "How many ETH Zurich spin-off companies were founded in 2024?"
context = "\n\n".join(retrieved_chunks)

answer = get_ai_response_with_context(query, context)
print(answer)

2025-09-18 11:10:16,578 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


In 2024, 37 ETH Zurich spin-off companies were founded.


## Assignments

### Extend a Query to a RAG Query

You want to know how many patent applications were reported at ETH Zurich in 2024. You ask the LLM:

In [ ]:
query = "How many patent applications were reported at ETH Zurich in 2024?"
answer = get_ai_response(query)
print(answer)

2025-09-18 11:10:21,089 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


I'm sorry, but I do not have real-time data access. I recommend checking the official website of ETH Zurich or contacting their technology transfer office for the most up-to-date information on patent applications in 2024.


**Task 1:** Fix the situation, using the same PDF about degree programs at ETH. In particular, you should:
* get a query embedding
* find the chunks closest to the query (in embedding space)
* build a list of the retrieved chunks, and add this to the context
* get an AI response for a query with the constructed context.

Note that all the necessary function calls are already available above - so you just need to find the relevant code and put it together.

**Solution:**

In [ ]:
query = "How many patent applications were reported at ETH Zurich in 2024?"
query_embedding = model.encode([query], normalize_embeddings=True)
D, I = index.search(query_embedding, k=2)  # get top-2 closest chunks

retrieved_chunks = [chunks[i] for i in I[0]]
context = "\n\n".join(retrieved_chunks)

answer = get_ai_response_with_context(query, context)
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-09-18 11:10:23,805 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


Based on the information provided, more than 100 patent applications were reported at ETH Zurich in 2024.


For better interpretability, you can also check the retrieved chunks:

In [ ]:
for id, chunk in enumerate(retrieved_chunks):
    print('=================================\nChunk ' + str(id+1) + ':')
    print(chunk)

Chunk 1:
More than patent applications 100

and more than 140 inventions reported in 2024

26 198 students of whom 4351 are doctoral students, from more than 120 countries

525 professors (  full-time equivalents, FTEs  )

33.3 % women students

97 %

of Master's graduates have a job after a year

## 1855

was the year in which the founders of modern Switzerland created this place of innovation and knowledge

6622 scientists

and 3344 technical and administrative staff
Chunk 2:
We are pleased that you are interested in a study programme at ETH Zurich. If you have a flair for figures and a liking for technology, and if you are interested in the natural sciences and the big challenges of the 21st century, then you are heading for the right place. ETH studies are based on wellfounded expert knowledge. That, in turn, will provide you with a basis on which you will be able to stay abreast of constant technological and scientific change. In addition, your studies at ETH


**(End of Solution)**

### Impact of `chunk_size` and `chunk_overlap` parameters
To get an intuition for the impact of the two parameters, you will experiment with the ```chunk_size``` and ```chunk_overlap``` parameters.

**Task 2:**

(a) Write a function that takes a cleaned Markdown string (we already have it!), chunk_size, chunk_overlap, and a query. It should return the retrieved chunks and the model’s answer. The function should do the following:

1. Split the cleaned Markdown into overlapping text chunks.

2. Encode the chunks with a SentenceTransformer.

3. Build a FAISS inner-product index (cosine similarity on L2-normalized vectors).

4. Retrieve the top-k chunks most similar to the query (use a reasonable default, e.g., k = 2).

5. Call a LLM helper to produce an answer from the (query + concatenated context).

(b) Examine the function’s behavior for different chunk sizes and overlaps. Describe your findings.

You may want to print or inspect the retrieved chunks to see how they affect the final answer.

**For your experiments, here are sample questions with their answers (based on the document)**:

1. How many ETH Zurich spin-off companies were founded in 2024? — 37

2. At what semester will ETH Zurich change its academic calendar? — Autumn semester 2027

3. How many patent applications were reported at ETH Zurich in 2024? — More than 100

4. What percentage of Master’s graduates at ETH Zurich have a job one year after graduation? — 97%

5. After Autumn Semester 2027, what will be the maximum permitted duration of studies? — 6 years

**Solution 2:**

Let us write the required function first:

In [ ]:
from typing import List, Tuple

def get_answer_with_context(
    chunk_size: int,
    chunk_overlap: int,
    clean_md: str,
    query: str,
) -> Tuple[List[str], str]:
    """
    Retrieve context-relevant chunks from a document and generate an answer.

    Args:
        chunk_size (int): Maximum number of characters per chunk.
        chunk_overlap (int): Overlap (in characters) between adjacent chunks.
        clean_md (str): The source document text (cleaned Markdown/plain text).
        query (str): The user question.

    Returns:
        Tuple[List[str], str]:
            - retrieved_chunks: the list of top retrieved text chunks.
            - answer: the model’s answer produced by `get_ai_response_with_context`.
    """

    # 1) Split the document into overlapping chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = text_splitter.split_text(clean_md)

    # 2) Embed chunks (normalize for cosine similarity)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeddings = model.encode(
        chunks,
        batch_size=64,
        show_progress_bar=True,
    )
    embeddings = embeddings.astype("float32")
    faiss.normalize_L2(embeddings)

    # 3) Build a FAISS index using inner product (equivalent to cosine on normalized vectors)
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    # 4) Encode the query and retrieve top-2 chunks
    query_embedding = model.encode([query], normalize_embeddings=True)
    D, I = index.search(query_embedding, k=2)  # get top-2 closest chunks

    retrieved_chunks = [chunks[i] for i in I[0]]
    context = "\n\n".join(retrieved_chunks)

    # 5) Produce the final answer using your LLM helper
    answer = get_ai_response_with_context(query, context)

    return retrieved_chunks, answer

---

Now we can start our experiments. Let's start with the following setting:

In [ ]:
query = "At what semester will ETH Zurich change its academic calendar?"

retrieved_chunks, answer = get_answer_with_context(200, 10, clean_md, query)

2025-09-18 11:10:35,228 - INFO - Use pytorch device_name: cuda
2025-09-18 11:10:35,229 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-09-18 11:10:37,030 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


In [ ]:
print(f"Question: {query}")
print(f"Retrieved chunks: {retrieved_chunks}")
print(f"Answer: {answer}")

Question: At what semester will ETH Zurich change its academic calendar?
Retrieved chunks: ['From the autumn semester 2027, ETH Zurich will change its academic calendar. The academic year will then consist of two semesters of practically the same length with the same exam preparation period', 'ethz.ch/which-study-programme\n\n## Applying\n\nYou can register or apply for a study programme at ETH Zurich from 1 December. Please note the relevant application deadlines.']
Answer: ETH Zurich will change its academic calendar starting from the autumn semester of 2027.


---

As we see, here we need only one sentence to get a correct answer. One may think we can try shortening the ```chunk_size```:

In [ ]:
retrieved_chunks, answer = get_answer_with_context(100, 10, clean_md, query)

2025-09-18 11:10:44,846 - INFO - Use pytorch device_name: cuda
2025-09-18 11:10:44,847 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-09-18 11:10:46,572 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


In [ ]:
print(f"Question: {query}")
print(f"Retrieved chunks: {retrieved_chunks}")
print(f"Answer: {answer}")

Question: At what semester will ETH Zurich change its academic calendar?
Retrieved chunks: ['From the autumn semester 2027, ETH Zurich will change its academic calendar. The academic year will', 'You can register or apply for a study programme at ETH Zurich from 1 December. Please note the']
Answer: ETH Zurich will change its academic calendar starting from the autumn semester of 2027.


---

The information provided is ok. Let's try another example:

In [ ]:
query = 'What percentage of Master’s graduates at ETH Zurich have a job one year after graduation?'
retrieved_chunks, answer = get_answer_with_context(100, 10, clean_md, query)

print(f"Question: {query}")
print(f"Retrieved chunks: {retrieved_chunks}")
print(f"Answer: {answer}")

2025-09-18 11:11:11,317 - INFO - Use pytorch device_name: cuda
2025-09-18 11:11:11,318 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-09-18 11:11:13,094 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


Question: What percentage of Master’s graduates at ETH Zurich have a job one year after graduation?
Retrieved chunks: ["97 %\n\nof Master's graduates have a job after a year\n\n## 1855", 'We are pleased that you are interested in a study programme at ETH Zurich. If you have a flair for']
Answer: Based on the provided information, 97% of Master's graduates at ETH Zurich have a job one year after graduation.


---

It was successful! Let's try decreasing ```chunk_overlap```:

In [ ]:
query = 'What percentage of Master’s graduates at ETH Zurich have a job one year after graduation?'
retrieved_chunks, answer = get_answer_with_context(100, 5, clean_md, query)

print(f"Question: {query}")
print(f"Retrieved chunks: {retrieved_chunks}")
print(f"Answer: {answer}")

2025-09-18 11:11:16,133 - INFO - Use pytorch device_name: cuda
2025-09-18 11:11:16,134 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-09-18 11:11:17,837 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


Question: What percentage of Master’s graduates at ETH Zurich have a job one year after graduation?
Retrieved chunks: ['your studies at ETH Zurich will provide you with the skill to reflect on your knowledge in various', 'We are pleased that you are interested in a study programme at ETH Zurich. If you have a flair for']
Answer: Based on the provided information, there is no specific percentage mentioned regarding the job placement of Master's graduates at ETH Zurich one year after graduation.


---

The information seems to be split not suitable for our question, such that it got into different chunks without enough overlap. Let's try to fix it by making the ```chunk_size``` higher (maybe each chunk will contain enough information to cover the information needed — maybe we can ignore the ```chunk_overlap``` parameter and just play with the ```chunk_size```?)

In [ ]:
query = 'What percentage of Master’s graduates at ETH Zurich have a job one year after graduation?'
retrieved_chunks, answer = get_answer_with_context(200, 5, clean_md, query)

print(f"Question: {query}")
print(f"Retrieved chunks: {retrieved_chunks}")
print(f"Answer: {answer}")

2025-09-18 11:11:23,733 - INFO - Use pytorch device_name: cuda
2025-09-18 11:11:23,734 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-09-18 11:11:25,576 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


Question: What percentage of Master’s graduates at ETH Zurich have a job one year after graduation?
Retrieved chunks: ["525 professors (  full-time equivalents, FTEs  )\n\n33.3 % women students\n\n97 %\n\nof Master's graduates have a job after a year\n\n## 1855", "## Master's studies (  180 credits  )\n\nFor the Master's studies, students move to a partner university in Basel, Lugano or Zurich, where their medical knowledge and skills are extended."]
Answer: Based on the provided information, 97% of Master's graduates at ETH Zurich have a job one year after graduation.


---

Now we were lucky: the chunk size was big enough to contain the information needed in one chunk. But still, it's a matter of split. If we make the ```chunk_size``` bigger a bit, it may cause not good split such that the information will be split not in our favor.

In [ ]:
query = 'What percentage of Master’s graduates at ETH Zurich have a job one year after graduation?'
retrieved_chunks, answer = get_answer_with_context(300, 5, clean_md, query)

print(f"Question: {query}")
print(f"Retrieved chunks: {retrieved_chunks}")
print(f"Answer: {answer}")

2025-09-18 11:11:29,632 - INFO - Use pytorch device_name: cuda
2025-09-18 11:11:29,632 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-09-18 11:11:31,520 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


Question: What percentage of Master’s graduates at ETH Zurich have a job one year after graduation?
Retrieved chunks: ['You can register or apply for a study programme at ETH Zurich from 1 December. Please note the relevant application deadlines.\n\nethz.ch/application-bachelor\n\n## Start', "«\n\nSelin Candan, is studying Human Medicine\n\n'I wanted to go to ETH   Zurich because it's a university with a personal touch. I saw the   newly introduced degree programme as an opportunity to profit from an innovative   curriculum.'\n\n## Master's studies (  180 credits  )"]
Answer: I'm sorry, but the provided information does not include data on the percentage of Master's graduates at ETH Zurich who have a job one year after graduation.


---
So we can't not take care about ```chunk_overlap```, it should be reasonably high. But not too high — more overlap → more chunks → higher processing time!

In [ ]:
query = 'What percentage of Master’s graduates at ETH Zurich have a job one year after graduation?'
retrieved_chunks, answer = get_answer_with_context(100, 10, clean_md, query)

print(f"Question: {query}")
print(f"Retrieved chunks: {retrieved_chunks}")
print(f"Answer: {answer}")

2025-09-18 11:11:37,814 - INFO - Use pytorch device_name: cuda
2025-09-18 11:11:37,814 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-09-18 11:11:39,566 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


Question: What percentage of Master’s graduates at ETH Zurich have a job one year after graduation?
Retrieved chunks: ["97 %\n\nof Master's graduates have a job after a year\n\n## 1855", 'We are pleased that you are interested in a study programme at ETH Zurich. If you have a flair for']
Answer: Based on the provided information, 97% of Master's graduates at ETH Zurich have a job one year after graduation.


---

Now let's have a look whether our choice of ```chunk_size``` 100 was good enough:

In [ ]:
query = 'After Autumn Semester 2027, what will be the maximum permitted duration of studies?'
retrieved_chunks, answer = get_answer_with_context(100, 10, clean_md, query)

print(f"Question: {query}")
print(f"Retrieved chunks: {retrieved_chunks}")
print(f"Answer: {answer}")

2025-09-18 11:11:43,159 - INFO - Use pytorch device_name: cuda
2025-09-18 11:11:43,159 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-09-18 11:11:45,108 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


Question: After Autumn Semester 2027, what will be the maximum permitted duration of studies?
Retrieved chunks: ['complete. The maximum permitted duration of studies is five years (six years as of the Autumn', 'Autumn Semester 2027).']
Answer: After Autumn Semester 2027, the maximum permitted duration of studies will be five years.


---

Unfortunately, no! The true answer is six. We see that the retrieved chunks aren't full enough. So we should keep the balance of ```chunk_size``` and ```chunk_overlap```:

In [ ]:
query = 'After Autumn Semester 2027, what will be the maximum permitted duration of studies?'
retrieved_chunks, answer = get_answer_with_context(200, 10, clean_md, query)

print(f"Question: {query}")
print(f"Retrieved chunks: {retrieved_chunks}")
print(f"Answer: {answer}")

2025-09-18 11:11:46,764 - INFO - Use pytorch device_name: cuda
2025-09-18 11:11:46,764 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-09-18 11:11:48,820 - INFO - HTTP Request: POST https://cas-dml-llm.openai.azure.com/openai/deployments/gpt-35-turbo/chat/completions?api-version=2024-05-01-preview "HTTP/1.1 200 OK"


Question: After Autumn Semester 2027, what will be the maximum permitted duration of studies?
Retrieved chunks: ["The Bachelor's degree programme comprises 180 credits and usually takes three years to complete. The maximum permitted duration of studies is five years (six years as of the Autumn Semester 2027).", 'The first two semesters form the first year of studies. Basic knowledge and fundamental skills are taught in the subjects of the first year. All subjects in the first year are compulsory.']
Answer: After the Autumn Semester 2027, the maximum permitted duration of studies will be six years for the Bachelor's degree programme, which comprises 180 credits and usually takes three years to complete.


---

Best combination of the ```chunk_size``` and ```chunk_overlap``` parameters should be chosen on the validation set of question.

In general, we see that
- ```chunk_size``` should be reasonably high to contain the information to answer the question. Too short chunk may be not enough to answer the question. But we can't make the chunk too big to avoid unnecessary high processing time.
- ```chunk_size``` shouldn't be too small. Text can be split in the chunks in such a way that the context needed will be split into several chunks, therefore we need some overlap. But not too big to avoid unnecessary high processing time.

**(End of Solution)**